# 06 — Semi-supervised Training: Pseudo-labeling

**Goal**: study how pseudo-labeling performs as a semi-supervised strategy compared to the
supervised baseline (nb02) and SimCLR fine-tuning (nb05), varying the fraction of labeled
data available at training time.

Figures → `reports/figures/06_semi_supervised/`  
CSV    → `reports/06_pseudo_labeling_results.csv`

## 0. Setup

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
        (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT / 'datasets').exists()):
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'src').exists() or not (PROJECT_ROOT / 'datasets').exists():
    raise RuntimeError('Cannot locate project root')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.semi_supervised.pseudo_labeling import train_pseudo_labeling
from src.utils import (
    fig_path, save_figure, save_results_csv,
    plot_confusion_matrix, plot_roc_curve,
)

REPORTS_DIR = str(PROJECT_ROOT / 'reports')
NB          = '06_semi_supervised'
DATA_ROOT   = str(PROJECT_ROOT / 'datasets' / 'raw')
CKPT_DIR    = str(PROJECT_ROOT / 'checkpoints')
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')
%matplotlib inline

## 1. Shared base config

All runs share the same architecture, split, augmentation and hyperparameters.
Only `labeled_ratio` changes across runs.

In [ ]:
BASE_CONFIG = {
    'data_root':            DATA_ROOT,
    'model_name':           'resnet18',
    'augmentation_name':    'weak',
    'epochs_per_round':     10,      # epochs of supervised training per round
    'num_rounds':           3,       # pseudo-labeling rounds
    'confidence_threshold': 0.95,    # min softmax confidence to accept a pseudo-label
    'batch_size':           32,      # same as nb02 ViT run for consistency
    'image_size':           224,
    'num_workers':          4,
    'split_ratios':         (0.7, 0.15, 0.15),
    'seed':                 42,      # same split as nb02
    'unlabeled_ratio':      0.5,     # fraction of train used as unlabeled pool
    'lr':                   1e-4,
    'device':               DEVICE,
    'checkpoint_dir':       CKPT_DIR,
}

# Labeled ratio sweep: 10%, 20%, 30% of the labeled training subset
LABELED_RATIOS = [0.1, 0.2, 0.3]

## 2. Sweep over labeled_ratio

For each value in `LABELED_RATIOS` we run `train_pseudo_labeling` and collect results.
Each run produces a checkpoint in `checkpoints/` and logs per-round val accuracy.

In [ ]:
all_results = {}  # labeled_ratio -> results dict

for lr_ratio in LABELED_RATIOS:
    print(f'\n{"="*60}')
    print(f'labeled_ratio = {lr_ratio:.0%}')
    print(f'{"="*60}')
    cfg = {**BASE_CONFIG, 'labeled_ratio': lr_ratio}
    results = train_pseudo_labeling(cfg)
    all_results[lr_ratio] = results
    print(f'  -> test_acc={results["test_accuracy"]:.4f} | '
          f'test_f1={results["test_f1"]:.4f} | test_auc={results["test_auc"]:.4f}')

print('\nAll runs completed.')

## 3. Per-round val accuracy

In [ ]:
# Plot val_acc per round for each labeled_ratio
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ['#4C72B0', '#DD8452', '#55A868']

for color, lr_ratio in zip(colors, LABELED_RATIOS):
    history = all_results[lr_ratio]['rounds_history']
    rounds  = [h['round'] for h in history]
    val_acc = [h['val_acc'] for h in history]
    labeled = [h['labeled_samples'] for h in history]
    ax.plot(rounds, val_acc, '-o', color=color, lw=2, ms=6,
            label=f'labeled={lr_ratio:.0%}')
    # annotate labeled set size at each round
    for r, v, l in zip(rounds, val_acc, labeled):
        ax.annotate(f'{l}', (r, v), textcoords='offset points',
                    xytext=(0, 8), ha='center', fontsize=7, color=color)

ax.set_xlabel('Pseudo-labeling round')
ax.set_ylabel('Val Accuracy')
ax.set_title('Val accuracy per round — pseudo-labeling sweep', fontsize=11)
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'rounds_val_acc.png'), show=True)

## 4. Pseudo-labels added per round

In [ ]:
# Stacked bar: how many pseudo-labels were added each round for each ratio
fig, axes = plt.subplots(1, len(LABELED_RATIOS), figsize=(4 * len(LABELED_RATIOS), 4),
                         sharey=False)

for ax, color, lr_ratio in zip(axes, colors, LABELED_RATIOS):
    history = all_results[lr_ratio]['rounds_history']
    rounds  = [h['round'] for h in history]
    added   = [h['pseudo_labeled_this_round'] for h in history]
    ax.bar(rounds, added, color=color, alpha=0.8)
    ax.set_title(f'labeled={lr_ratio:.0%}', fontsize=9)
    ax.set_xlabel('Round')
    ax.set_ylabel('Pseudo-labels added')
    ax.set_xticks(rounds)
    ax.grid(axis='y', alpha=0.3)
    for r, a in zip(rounds, added):
        ax.text(r, a + max(added)*0.01, str(a), ha='center', fontsize=8)

fig.suptitle('Pseudo-labels added per round', fontsize=11)
fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'pseudo_labels_per_round.png'), show=True)

## 7. Confusion matrix — best labeled_ratio run

In [ ]:
# Use the run with the highest test accuracy
best_ratio = max(all_results, key=lambda k: all_results[k]['test_accuracy'])
best_res   = all_results[best_ratio]
print(f'Best labeled_ratio: {best_ratio:.0%} | acc={best_res["test_accuracy"]:.4f}')

plot_confusion_matrix(
    best_res['test_y_true'], best_res['test_y_pred'],
    title=f'Confusion Matrix — Pseudo-labeling (labeled={best_ratio:.0%})',
    path=fig_path(REPORTS_DIR, NB, f'cm_best_ratio.png'),
    show=True,
)

## 8. ROC curves

In [ ]:
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(6, 5))

for color, lr_ratio in zip(colors, LABELED_RATIOS):
    res = all_results[lr_ratio]
    fpr, tpr, _ = roc_curve(res['test_y_true'], res['test_y_prob'])
    ax.plot(fpr, tpr, lw=2, color=color,
            label=f"labeled={lr_ratio:.0%} (AUC={res['test_auc']:.3f})")

ax.plot([0,1], [0,1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Pseudo-labeling sweep', fontsize=11)
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
fig.tight_layout()
save_figure(fig, fig_path(REPORTS_DIR, NB, 'roc_sweep.png'), show=True)